In [13]:
# Load libraries
library(tidyverse)
library(dplyr)
library(GGally)
library(broom)

# Adjust plot and font sizes
options(repr.plot.width = 15, repr.plot.height = 12)
set_theme(theme_gray(base_size = 16))

In [14]:
wc_dat <- read_csv(file = "players-selected-columns.csv")

# Remove players who've not played

wc_dat <- wc_dat |> filter(minutes > 0)

# Group hybrid players (e.g. "FW,MF" and "MF,FW") together

wc_dat <- wc_dat |> mutate(position = sapply(strsplit(position, ","), function(x) { paste(sort(x), collapse = ",")  })) 

# Remove team_country, shots per EDA
wc_dat <- wc_dat |> mutate(team_country = NULL, shots = NULL)

Rows: 1248 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): team_country, position
dbl (11): age, minutes, goals, assists, cards_yellow, shots, shots_on_target...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Model 1: Linear Regression

## TODO: Do LR once we settle on how to approach this

### Model 2: Logistic Regression

In [15]:
# Add new column indicate if player has scored or not

has_scored_dat <- if_else(wc_dat$goals > 0, 1, 0)

wc_dat <- wc_dat |> mutate(has_scored = has_scored_dat)

head(wc_dat)

position,age,minutes,goals,assists,cards_yellow,shots_on_target,fouls,offsides,interceptions,tackles_won,has_scored
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
FW,23,19,0,0,0,0,0,1,0,0,0
FW,26,251,1,0,0,2,1,0,0,1,1
MF,24,98,0,0,0,0,0,0,1,0,0
DF,34,360,0,0,0,0,0,0,4,3,0
MF,23,360,0,0,1,3,3,1,3,5,0
"FW,MF",23,1,0,0,0,0,0,0,0,0,0


In [18]:
# Logreg model

wc_log_model <- glm(formula = has_scored ~ . - goals, data = wc_dat, family = "binomial")

# summarize(wc_log_model)

wc_log_model_stat <- wc_log_model |> tidy(exponentiate = TRUE) |> mutate_if(is.numeric, round, 3)

wc_log_model_stat

term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.069,0.844,-3.166,0.002
"positionDF,FW",0.000,2282.397,-0.006,0.995
"positionDF,MF",0.837,0.687,-0.259,0.796
positionFW,1.620,0.412,1.172,0.241
"positionFW,MF",1.082,0.464,0.170,0.865
positionGK,0.000,500.499,-0.028,0.977
positionMF,1.444,0.312,1.178,0.239
age,0.974,0.029,-0.921,0.357
minutes,1.000,0.001,0.342,0.732
